<a href="https://colab.research.google.com/github/INTROESPACIAL20261/pantilla0/blob/main/lima_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Instalar
!pip -q install geopandas pyogrio libpysal esda folium mapclassify branca

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
from pathlib import Path

import branca
import esda
import folium
import geopandas as gpd
import libpysal
import numpy as np
import pandas as pd
from IPython.display import display, HTML

DATA_URL = "https://github.com/CienciaDeDatosEspacial/dataSets/raw/refs/heads/main/PERU/PeruMaps.gpkg"
VALUE_COL = "Educ_sec_comp2019_pct"
ALPHA = 0.05

LABELS = {
    "Insignificant": "3 no_pattern",
    "Low-Low": "4 coldSpot",
    "High-High": "2 hotSpot",
    "High-Low": "1 hotOutlier",
    "Low-High": "5 coldOutlier",
}

LABEL_ORDER = ["1 hotOutlier", "2 hotSpot", "3 no_pattern", "4 coldSpot", "5 coldOutlier"]
COLORS = {
    "1 hotOutlier": "#b35806",
    "2 hotSpot": "#f1a340",
    "3 no_pattern": "#e6e6e6",
    "4 coldSpot": "#998ec3",
    "5 coldOutlier": "#542788",
}

In [ ]:
peru = gpd.read_file(DATA_URL, layer="good_geom", engine="pyogrio")
lima = peru.loc[peru["DEPARTAMENTO"].str.upper().eq("LIMA")].copy()
lima = lima.loc[lima[VALUE_COL].notna()].reset_index(drop=True)
lima = lima.to_crs(4326)
lima_projected = lima.to_crs(32718)

print(f"Distritos de Lima usados: {len(lima)}")
lima[["DEPARTAMENTO", "PROVINCIA", "DISTRITO", VALUE_COL]].head()

In [ ]:
def distance_threshold_for_no_islands(gdf_projected):
    coords = np.column_stack([gdf_projected.geometry.centroid.x, gdf_projected.geometry.centroid.y])
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=2).fit(coords)
    distances, _ = nn.kneighbors(coords)
    return float(np.ceil(distances[:, 1].max() / 1000) * 1000)


distance_threshold = distance_threshold_for_no_islands(lima_projected)

weights = {
    "Queen / interseccion": libpysal.weights.Queen.from_dataframe(lima_projected, use_index=False),
    "Rook / frontera compartida": libpysal.weights.Rook.from_dataframe(lima_projected, use_index=False),
    "KNN 4 centroides": libpysal.weights.KNN.from_dataframe(lima_projected, k=4),
    f"Banda distancia {distance_threshold/1000:.0f} km": libpysal.weights.DistanceBand.from_dataframe(
        lima_projected, threshold=distance_threshold, binary=True, silence_warnings=True
    ),
}

for w in weights.values():
    w.transform = "r"

[(name, round(np.mean(list(w.cardinalities.values())), 2), len(w.islands)) for name, w in weights.items()]

In [ ]:
def clean_prefix(name):
    import re
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


def lisa_for_weight(gdf, name, w):
    prefix = clean_prefix(name)
    lisa = esda.Moran_Local(gdf[VALUE_COL].astype(float), w, permutations=999, seed=42)
    labels = pd.Series(lisa.get_cluster_labels(crit_value=ALPHA)).replace(LABELS).to_numpy()
    out = gdf.copy()
    out[f"{prefix}_lisa"] = labels
    out[f"{prefix}_moran_i"] = lisa.Is
    out[f"{prefix}_p"] = lisa.p_sim
    out[f"{prefix}_n"] = [w.cardinalities[i] for i in range(len(gdf))]
    summary = {
        "vecindario": name,
        "prefix": prefix,
        "vecinos_promedio": np.mean(list(w.cardinalities.values())),
        "islas": len(w.islands),
    }
    summary.update(out[f"{prefix}_lisa"].value_counts().reindex(LABEL_ORDER, fill_value=0).to_dict())
    return out, summary


results = {}
summaries = []
for name, w in weights.items():
    result, summary = lisa_for_weight(lima, name, w)
    results[summary["prefix"]] = result
    summaries.append(summary)

summary_df = pd.DataFrame(summaries)
summary_df

In [ ]:
def style_function(prefix):
    def _style(feature):
        label = feature["properties"][f"{prefix}_lisa"]
        return {
            "color": "#303030",
            "weight": 0.35,
            "fillColor": COLORS.get(label, "#e6e6e6"),
            "fillOpacity": 0.45 if label == "3 no_pattern" else 0.78,
        }
    return _style


def tooltip_fields(prefix):
    return ["PROVINCIA", "DISTRITO", VALUE_COL, f"{prefix}_lisa", f"{prefix}_p", f"{prefix}_n"]


def make_map(gdf, prefix, title):
    m = folium.Map(location=[gdf.geometry.union_all().centroid.y, gdf.geometry.union_all().centroid.x], zoom_start=7, tiles="cartodbpositron")
    folium.GeoJson(
        gdf,
        name=title,
        style_function=style_function(prefix),
        tooltip=folium.GeoJsonTooltip(
            fields=tooltip_fields(prefix),
            aliases=["Provincia", "Distrito", VALUE_COL, "Clase LISA", "p-value", "Vecinos"],
            localize=True,
        ),
    ).add_to(m)
    legend_html = "<div style='position: fixed; bottom: 20px; left: 20px; z-index: 9999; background: white; padding: 10px; border: 1px solid #999; font-size: 12px;'>"
    legend_html += f"<b>{title}</b><br>"
    for label in LABEL_ORDER:
        legend_html += f"<span style='display:inline-block;width:12px;height:12px;background:{COLORS[label]};border:1px solid #555;'></span> {label}<br>"
    legend_html += "</div>"
    m.get_root().html.add_child(branca.element.Element(legend_html))
    return m


# Ultimo mapa LISA equivalente al notebook original: Queen/interseccion, solo Lima.
last_map_prefix = "queen_interseccion"
mapa_lisa_lima = make_map(results[last_map_prefix], last_map_prefix, "LISA Lima - Queen / interseccion")
mapa_lisa_lima

In [ ]:
def panel_html(prefix, title):
    html_path = f"map_{prefix}.html"
    make_map(results[prefix], prefix, title).save(html_path)
    with open(html_path, "r", encoding="utf-8") as f:
        srcdoc = f.read().replace('"', "&quot;")
    return f'''
      <section class="panel">
        <h2>{title}</h2>
        <iframe srcdoc="{srcdoc}"></iframe>
      </section>
    '''


summary_html = summary_df.drop(columns=["prefix"]).to_html(index=False, classes="summary-table")
panels = []
for item in summaries:
    panels.append(panel_html(item["prefix"], item["vecindario"]))

dashboard_html = f'''
<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<title>Dashboard LISA Lima</title>
<style>
body {{ margin: 0; font-family: Arial, sans-serif; background: #f7f8fa; color: #202124; }}
header {{ background: white; padding: 18px 24px; border-bottom: 1px solid #d7dce2; }}
h1 {{ margin: 0 0 8px; font-size: 24px; }}
p {{ margin: 0; color: #5f6368; }}
.summary {{ background: white; padding: 14px 24px; overflow-x: auto; border-bottom: 1px solid #d7dce2; }}
.summary-table {{ border-collapse: collapse; width: 100%; font-size: 13px; }}
.summary-table th, .summary-table td {{ border-bottom: 1px solid #edf0f2; padding: 7px 8px; text-align: right; }}
.summary-table th:first-child, .summary-table td:first-child {{ text-align: left; }}
main {{ padding: 16px; display: grid; grid-template-columns: repeat(2, minmax(320px, 1fr)); gap: 16px; }}
.panel {{ background: white; border: 1px solid #d7dce2; border-radius: 8px; overflow: hidden; }}
.panel h2 {{ height: 42px; line-height: 42px; margin: 0; padding: 0 12px; font-size: 15px; border-bottom: 1px solid #d7dce2; }}
iframe {{ width: 100%; height: 430px; border: 0; display: block; }}
@media (max-width: 900px) {{ main {{ grid-template-columns: 1fr; }} }}
</style>
</head>
<body>
<header>
  <h1>Dashboard LISA - Distritos del Departamento de Lima</h1>
  <p>Misma data y variable; cuatro calculos de vecindarios para resaltar cambios en coldSpot, hotSpot, hotOutlier y coldOutlier.</p>
</header>
<section class="summary">{summary_html}</section>
<main>{''.join(panels)}</main>
</body>
</html>
'''

Path("dashboard_lisa_lima_4_vecindarios.html").write_text(dashboard_html, encoding="utf-8")
summary_df.to_csv("resumen_lisa_lima_4_vecindarios.csv", index=False, encoding="utf-8-sig")

print("Archivos creados en Colab:")
print("- dashboard_lisa_lima_4_vecindarios.html")
print("- resumen_lisa_lima_4_vecindarios.csv")